### Documents

In [1]:
# 1. load documents
from pathlib import Path
import requests

In [ ]:
RFCS = {
    "RFC768_UDP.txt": 768,
    "RFC791_IPv4.txt": 791,
    "RFC792_ICMP.txt": 792,
    "RFC793_TCP.txt": 793,
    "RFC826_ARP.txt": 826,
    "RFC1034_DNS_Concepts.txt": 1034,
    "RFC1035_DNS_Implementation.txt": 1035,
    "RFC2328_OSPFv2.txt": 2328,
    "RFC2453_RIPv2.txt": 2453,
    "RFC4271_BGP4.txt": 4271,
    "RFC4861_IPv6_ND.txt": 4861,
}

output = Path("docs")
output.mkdir(exist_ok=True)

for filename, rfc in RFCS.items():
    url = f"https://www.rfc-editor.org/rfc/rfc{rfc}.txt"
    print(f"Downloading RFC {rfc}...")
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    (output / filename).write_text(response.text, encoding="utf-8")

print("Done!")

## Create Embeddings and Store in Chroma

In [7]:
import os
import sys
import chromadb
from typing import List, Any

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader


In [3]:
load_dotenv(override=True)
env_path = os.getenv("BASE_PATH")
# Including for import modules in root's project directory 
if str(Path.cwd().parent) not in sys.path:
    sys.path.append(str(Path.cwd().parent))

In [20]:
def load_documents_selector(data_dir: str) -> List[Any]:
    """
    Select data type for loading from respective directory convert to LangChain document structure.
    Supported type: PDF, TXT, CSV
    """

    # Root project directory
    BASE_PATH= Path("./docs/").resolve()
    documents = []

    if data_dir == "pdf":
        # PDF files
        pdf_files = list(BASE_PATH.glob('**/*.pdf'))
        for pdf_file in pdf_files:
            try:
                loader = PyPDFLoader(str(pdf_file))
                loaded = loader.load()
                documents.extend(loaded)
            except Exception as e:
                print(f"[ERROR] Failed to load PDF {pdf_file}: {e}")
    elif data_dir == "txt":
        # TXT files
        txt_files = list(BASE_PATH.glob('**/*.txt'))
        for txt_file in txt_files:
            try:
                loader = TextLoader(str(txt_file))
                loaded = loader.load()
                for doc in loaded:
                    source = Path(doc.metadata["source"])
                    doc.metadata["source"] = str(source.relative_to(BASE_PATH))
                documents.extend(loaded)
            except Exception as e:
                print(f"[ERROR] Failed to load TXT {txt_file}: {e}")
    elif data_dir == "csv":
        # CSV files
        csv_files = list(BASE_PATH.glob('**/*.csv'))
        for csv_file in csv_files:
            target_metadata = ['country_name','iso_country','region_name','local_region','icao_code','iata_code']
            try:
                loader = CSVLoader(
                    file_path=str(csv_file),
                    encoding='utf-8',
                    metadata_columns=target_metadata
                )
                loaded = loader.load()
                for doc in loaded:
                    source = Path(doc.metadata["source"])
                    doc.metadata["source"] = str(source.relative_to(BASE_PATH))
                documents.extend(loaded)
            except Exception as e:
                print(f"[ERROR] Failed to load CSV {csv_file}: {e}")
    else:
        print(f"[ERROR] directory {data_dir} does not exit")

    print(f"Total loaded documents: {len(documents)}")
    return documents


In [23]:
documents = load_documents_selector('txt')

Total loaded documents: 11


In [28]:
def chunk_documents(documents, chunk_size=1000, chunk_overlap=200):
    """
    Split documents into smaller chunks for better retrieval
    
    Args:
        documents: List of Document objects or single Document
        chunk_size: Size of each chunk (characters)
        chunk_overlap: Overlap between chunks to maintain context
    """
    # Initialize text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ".", " ", ""],
        keep_separator=False,
        add_start_index=True,
    )
    
    # If single document, convert to list
    if isinstance(documents, Document):
        documents = [documents]
    
    # Split documents
    chunks = text_splitter.split_documents(documents)
    
    print(f"Created {len(chunks)} chunks from {len(documents)} documents")
    return chunks

In [30]:
chunks = chunk_documents(documents)

Created 2235 chunks from 11 documents


In [36]:
def initialize_chroma_db(collection_name=None, delete_existing=False, base_path=None):
    """
    Initialize (or reuse) a persistent ChromaDB collection, using a path
    built from the BASE_PATH environment variable.

    Flow:
      1. Resolve directory path from BASE_PATH env var (or `base_path` override).
      2. Check if the directory exists.
         2.1 If not, create it.
      3. Check if the collection already exists.
         3.1 If it exists, it is kept by default (delete_existing=False).
             It's only deleted+recreated if delete_existing=True is passed.
         If it doesn't exist yet, it's created fresh.

    Args:
        collection_name (str): Name of the collection to check/create.
                                Defaults to "chroma_db" if not provided.
        delete_existing (bool): If True, deletes an existing collection before
                                 recreating it. Defaults to False (safe default).
        base_path (str): Optional override for BASE_PATH env var (mainly for
                          testing). If not given, reads os.getenv("BASE_PATH").

    Returns:
        chromadb.api.models.Collection.Collection: The ready-to-use collection.

    Raises:
        ValueError: If BASE_PATH is not set and no base_path override is given.
        PermissionError: If the directory can't be created due to permissions.
        OSError: For other filesystem-related failures.
        RuntimeError: If the Chroma client/collection fails to initialize
                      (e.g. corrupted store).
    """
    db_name = "chroma_db"
    collection_name = collection_name or db_name

    # 1. Resolve base path
    env_path = base_path if base_path is not None else os.getenv("BASE_PATH")
    if not env_path:
        raise ValueError(
            "BASE_PATH environment variable is not set and no base_path was provided. "
            "Set it with os.environ['BASE_PATH'] = '/some/path/' or pass base_path explicitly."
        )

    # Guard against missing trailing slash (env_path + 'chroma_db' would otherwise
    # merge into the parent folder name, e.g. '/data' + 'chroma_db' -> '/datachroma_db')
    db_path = os.path.join(env_path, db_name)

    # 2. Check / create directory
    try:
        if not os.path.exists(db_path):
            os.makedirs(db_path, exist_ok=True)
            print(f"Directory '{db_path}' did not exist. Created it.")
        else:
            print(f"Directory '{db_path}' already exists.")
    except PermissionError as e:
        raise PermissionError(
            f"No permission to create directory at '{db_path}'. "
            f"Check filesystem permissions or BASE_PATH value."
        ) from e
    except OSError as e:
        raise OSError(f"Failed to create directory '{db_path}': {e}") from e

    # 3. Connect to persistent client
    try:
        client = chromadb.PersistentClient(path=db_path)
    except Exception as e:
        raise RuntimeError(
            f"Failed to initialize Chroma persistent client at '{db_path}'. "
            f"The store may be corrupted or incompatible with the installed "
            f"chromadb version. Original error: {e}"
        ) from e

    # 4. Check if the collection already exists
    try:
        existing_names = [c.name for c in client.list_collections()]
    except Exception as e:
        raise RuntimeError(f"Failed to list existing collections: {e}") from e

    collection_exists = collection_name in existing_names

    try:
        if collection_exists:
            if delete_existing:
                client.delete_collection(name=collection_name)
                print(f"Collection '{collection_name}' existed and was deleted (delete_existing=True).")
                collection = client.create_collection(name=collection_name)
                print(f"Collection '{collection_name}' recreated.")
            else:
                collection = client.get_collection(name=collection_name)
                print(f"Collection '{collection_name}' already exists. Reusing it.")
        else:
            collection = client.create_collection(name=collection_name)
            print(f"Collection '{collection_name}' did not exist. Created it.")
    except Exception as e:
        raise RuntimeError(
            f"Failed to get/create/delete collection '{collection_name}': {e}"
        ) from e

    return collection


# Example usage:
# os.environ["BASE_PATH"] = "/data/"
# collection = initialize_chroma_db()                          # keeps existing collection
# collection = initialize_chroma_db(delete_existing=True)       # wipes and recreates

In [39]:
collection = initialize_chroma_db()

Directory './chroma_db' already exists.
Collection 'chroma_db' already exists. Reusing it.


In [44]:
def get_ollama_embeddings(model: str = "embeddinggemma") -> OllamaEmbeddings:
    """
    Local embeddings via Ollama.
    Make sure `ollama serve` is running and the model is pulled:
    `ollama pull embeddinggemma`
    """
    return OllamaEmbeddings(
        model=model,
    )

In [40]:
# 3. Initialize ChromaDB + add chunks (unified function)

def build_vectorstore(chunks, embedding_fn, collection_name: str):
    """
    Create (or append to) a persistent Chroma collection
    and store the embedded chunks.
    """
    vectorstore = Chroma(
        collection_name=collection_name,
        embedding_function=embedding_fn,
        persist_directory="./chroma_db",
    )

    BATCH = 200
    for i in range(0, len(chunks), BATCH):
        batch = chunks[i:i + BATCH]
        vectorstore.add_documents(batch)
        print(f"  Added {i+len(batch)}/{len(chunks)}")

    print(f"✅ Stored {len(chunks)} chunks in collection '{collection_name}'")
    return vectorstore

In [45]:
ollama_vs = build_vectorstore(
    chunks,
    embedding_fn=get_ollama_embeddings("mxbai-embed-large"),
    collection_name="gemma_local2",
)

  Added 200/2235
  Added 400/2235
  Added 600/2235
  Added 800/2235
  Added 1000/2235
  Added 1200/2235
  Added 1400/2235
  Added 1600/2235
  Added 1800/2235
  Added 2000/2235
  Added 2200/2235
  Added 2235/2235
✅ Stored 2235 chunks in collection 'gemma_local2'


In [46]:
retriever = ollama_vs.as_retriever(search_kwargs={"k": 5})
relevant_chunks = retriever.invoke("What is TCP?")
len(relevant_chunks)

5

### Visualize DB

In [47]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [ ]:
CHROMA_PATH = "./chroma_db"

COLLECTION_NAME = "gemma_local2"     # match your collection_name
EMBEDDING_MODEL = "mxbai-embed-large"

def load_vectors(collection_name: str, persist_directory: str, embedding_model: str):
    """Pull raw vectors + text + metadata out of an existing Chroma collection."""
    vs = Chroma(
        collection_name=collection_name,
        embedding_function=OllamaEmbeddings(model=embedding_model),
        persist_directory=persist_directory,
    )
 
    # include=["embeddings"] is what actually gets you the raw vectors back
    result = vs.get(include=["embeddings", "documents", "metadatas"])
 
    vectors = np.array(result["embeddings"])
    documents = result["documents"]
    metadatas = result["metadatas"]
 
    print(f"Loaded {vectors.shape[0]} vectors of dimension {vectors.shape[1]}")
    return vectors, documents, metadatas

def reduce_dimensions(vectors: np.ndarray, n_components: int, method: str = "pca"):
    """Project high-dim vectors down to n_components (2 or 3) for plotting."""
    if method == "pca":
        reducer = PCA(n_components=n_components, random_state=42)
    elif method == "tsne":
        # perplexity must be < n_samples; keep it modest for small datasets
        perplexity = min(30, max(5, vectors.shape[0] // 3))
        reducer = TSNE(n_components=n_components, perplexity=perplexity, random_state=42)
    else:
        raise ValueError("method must be 'pca' or 'tsne'")
 
    return reducer.fit_transform(vectors)

def make_hover_text(documents, max_len: int = 120):
    """Short preview of each chunk for hover tooltips."""
    return [
        (doc[:max_len] + "...") if len(doc) > max_len else doc
        for doc in documents
    ]
 
 
def plot_2d(coords: np.ndarray, documents, source_labels=None, title: str = "Embeddings (2D)"):
    hover = make_hover_text(documents)
    fig = px.scatter(
        x=coords[:, 0],
        y=coords[:, 1],
        color=source_labels,  # e.g. color by source file, if you have metadata for it
        hover_name=hover,
        title=title,
    )
    fig.update_traces(marker=dict(size=8, opacity=0.75))
    fig.show()
 
 
def plot_3d(coords: np.ndarray, documents, source_labels=None, title: str = "Embeddings (3D)"):
    hover = make_hover_text(documents)
    fig = px.scatter_3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        color=source_labels,
        hover_name=hover,
        title=title,
    )
    fig.update_traces(marker=dict(size=4, opacity=0.75))
    fig.show()

In [49]:
vectors, documents, metadatas = load_vectors(COLLECTION_NAME, CHROMA_PATH, EMBEDDING_MODEL)

# Optional: color points by source document, if that metadata exists
source_labels = [m.get("source", "unknown") for m in metadatas] if metadatas else None

Loaded 2235 vectors of dimension 1024


In [50]:
# --- 2D ---
coords_2d = reduce_dimensions(vectors, n_components=2, method="pca")
plot_2d(coords_2d, documents, source_labels, title="Chunks - PCA (2D)")

In [51]:
# --- Optional: t-SNE tends to show clusters more clearly than PCA ---
coords_2d_tsne = reduce_dimensions(vectors, n_components=2, method="tsne")
plot_2d(coords_2d_tsne, documents, source_labels, title="Chunks - t-SNE (2D)")

In [52]:
# --- 3D ---
coords_3d = reduce_dimensions(vectors, n_components=3, method="pca")
plot_3d(coords_3d, documents, source_labels, title="Chunks - PCA (3D)")

## Generation with RAG contexts

In [4]:
from src.intelligence_v3 import Conversation, create_provider, Intelligence, ChromaRetriever

In [5]:
CHROMA_PATH = "".join([env_path,'chroma_db/'])

if os.path.exists(CHROMA_PATH):
    files = os.listdir(CHROMA_PATH)
    print(f"Files in {CHROMA_PATH}:")
    for f in files:
        print(f" - {f}")
else:
    print("Directory does not exist.")

Files in ./chroma_db/:
 - 99cf17c5-4940-41ca-a72d-3adebf163702
 - chroma.sqlite3


In [8]:
COLLECTION_NAME = 'gemma_local2'

vectorstore = Chroma(
      collection_name=COLLECTION_NAME,
      embedding_function=OllamaEmbeddings(model="mxbai-embed-large"),
      persist_directory=CHROMA_PATH,
  )

In [9]:
retriever = ChromaRetriever(vectorstore, default_k=5)
rag_provider = create_provider("gemma-4-31b-it")
rag_llm = Intelligence(
        rag_provider,
        system_prompt="Answer questions using only the provided context.",
        retriever=retriever,
    )
rag_conversation = Conversation()
rag_conversation.system(rag_llm.system_prompt)

In [10]:
answer = rag_llm.ask(
        "What does an ICMP Redirect message do, and when is it sent?",
        conversation=rag_conversation,
    )
print(answer)

ICMP Redirect messages are sent by routers to:
* Redirect a host to a better first-hop router (or node) for a specific destination [1], [4].
* Inform hosts that a destination is actually a neighbor (i.e., on-link) [1], [4].


In [11]:
# Follow-up: retrieval runs again on this new question, but the
# conversation history stays clean (no duplicated context blocks)
follow_up_rag = rag_llm.ask(
        "And what about the BGP?",
        conversation=rag_conversation,
    )
print(follow_up_rag)

The Border Gateway Protocol (BGP) is an inter-Autonomous System routing protocol [1], [3]. Its primary function is to exchange network reachability information with other BGP systems [1], [3]. 

This reachability information includes a list of Autonomous Systems (ASes) that the information traverses, which is used to:
* Construct a graph of AS connectivity [1], [3].
* Prune routing loops [1], [3].
* Enforce some policy decisions at the AS level [1], [3].

Additionally, BGP was built on experience gained from EGP and EGP usage in the NSFNET Backbone [3]. In the context of the provided text, a BGP speaker is assumed to advertise only the routes it uses itself, meaning the most preferred BGP route used in forwarding [2].


### Evaluation  

```
run_real_eval.py
  │
  │ 1. builds retriever + provider from src.intelligence_v3
  │    (ChromaRetriever, create_provider, format_context, RAG_PROMPT_TEMPLATE)
  │
  │ 2. defines my_rag_pipeline(question) -> RagResult
  │    (this is the ONLY function rag_eval calls into your app)
  │
  ▼
rag_eval.evaluator.run_evaluation(csv_path, my_rag_pipeline, out_path=...)
  │
  │ 3. loader.load_eval_csv(csv_path) -> list[EvalCase]
  │
  │ 4. for each EvalCase:
  │      evaluator._call_with_retry(my_rag_pipeline, case.question)
  │        -> calls INTO src.intelligence_v3 (your retriever + LLM)
  │        -> returns RagResult(retrieved_chunks, generated_answer)
  │
  │ 5. metrics.py scores that RagResult against the EvalCase's
  │      gold_answer / source_doc / source_keywords / assertions
  │      (mapping.py resolves source_doc <-> RFC ids along the way)
  │
  │ 6. writes one row to eval_results.csv IMMEDIATELY (incremental save)
  │
  ▼
report.py summarize() / diagnose() / print_report()
  -> aggregates all CaseResults into the console report
```

### Run module **"run_real_eval.py"**

#### ----  Real config ----  
1- **CHROMA_PATH** relative to root project such as "./chroma_db"  
2- **COLLECTION_NAME** ChromaDB collection "gemma_local2"  
3- **vectorstore** Load vectors from Chroma DB  
4- **retriever** Query k chunks from Chroma DB  
5- **rag_provider** Create provider for evaluation such **"gemma-4-31b-it"** or **"gpt-5.4-nano"**
6- **SYSTEM_PROMPT** Intend that LLM limits his answer only to the provided context  

### Running it  
python run_real_eval.py

## Evaluation result  
**Tests implemented** and how to read their output is detailed in project_documentation folder/**.md** 